In [ ]:
# The Jupyter Notebook where I experiment on:
# Improving the initial state for TU3

# Tudat imports for propagation and estimation
from tudatpy.interface import spice
from tudatpy.dynamics import environment_setup, parameters_setup, propagation_setup
from tudatpy import estimation
from tudatpy.estimation import observable_models_setup, estimation_analysis
from tudatpy.constants import GRAVITATIONAL_CONSTANT
from tudatpy.astro.frame_conversion import inertial_to_rsw_rotation_matrix
import matplotlib.gridspec as gridspec
from tudatpy.data.mpc import BatchMPC
from tudatpy.data.horizons import HorizonsQuery
from tudatpy.data.sbdb import SBDBquery


# other useful modules
import numpy as np
import datetime
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.cm as cm
from tudatpy.astro import time_representation
from tudatpy.astro.time_representation import DateTime
from astropy.table import Table
from tudatpy.astro import element_conversion    # for TU3 initial state

# additional things for the asteroids
from tudatpy import constants
import os           # for the extraction of asteroid kernels


# SPICE KERNELS
spice.load_standard_kernels()

In [ ]:
# Defining some constants

# Target 1998 TU3 (66146)
target_mpc_code = "66146"

# 10 years
observations_start = datetime.datetime(2015, 1, 1)      # 1st Jan 2015 at 12:00
observations_end = datetime.datetime(2025, 1, 1)

# number of iterations for our estimation
number_of_pod_iterations = 6

# timestep of 24 hours for our estimation
timestep_global = 24 * 3600.0

# 2 month time buffer used to avoid interpolation errors:
time_buffer = 2 * 31 * 86400.0

# # define the frame origin and orientation.
# global_frame_origin = "SSB"
# global_frame_orientation = "J2000"

In [5]:
# Supposedly codes_300ast_20100725.bsp contains many asteroids

target_sbdb = SBDBquery(target_mpc_code)

mpc_codes = [target_mpc_code]  # the BatchMPC interface requires a list.
target_spkid = target_sbdb.codes_300_spkid  # the ID used by Tudat

print(target_sbdb.query["object"].keys())
obj = target_sbdb.query["object"]

# The ID used by Tudat (biggest asteroids have shortname, but smaller don't)
target_name = (
    obj.get("shortname") or
    obj.get("fullname") or
    obj.get("des")
)  

print(f"SPK ID for {target_name} is: {target_spkid}")

odict_keys(['orbit_class', 'prefix', 'des', 'neo', 'pha', 'orbit_id', 'kind', 'spkid', 'fullname'])
SPK ID for 66146 (1998 TU3) is: 2066146


In [12]:
# # TU3 Ephemeris Uncertainty
# # I will skip this for now cause it's more complicated for smaller asteroids

# import requests

# url = "https://ssd-api.jpl.nasa.gov/sbdb.api?des=1998%20TU3&cov=mat"
# data = requests.get(url).json()

# cov = data["orbit"]["covariance"]

# print(cov.keys())

In [ ]:
# Sure let's try 3 different acceleration levels

# Level 1: Sun, 8 planets and GR
# Level 2: Sun, 8 planets, 21 asteroids and GR
# Level 3: Sun, 8 planets, 21 asteroids, GR, J2 and Yarkovsky

setup_names = ["LVL1 Accelerations", "LVL2 Accelerations", "LVL3 Accelerations"]

accel_levels = [1, 2, 3]
use_sat_data = [False, False, False]
use_catalog_cor = [False, False, False]
use_weighting = [False, False, False]

satellites_names = ["WISE"]
satellites_MPC_codes = ["C51"]  # C51 is the observatory code MPC uses for WISE
satellites_Horizons_codes = [
    "-163"
]  # -163 is the query ID for WISE in Horizons see explanation below.


# Consider trying out different combinations of satellites.
# Note that you must change the dates to use TESS as it launched in April 2018
# satellites_names = ["WISE", "TESS"]
# satellites_MPC_codes = ["C51", "C57"]
# satellites_Horizons_codes = ["-163", "-95"]

In [24]:
# Retrieving the observations

batch = BatchMPC()
batch.get_observations(mpc_codes)
batch.filter(
    epoch_start=observations_start,
    epoch_end=observations_end,
)

print(batch.summary())


   Batch Summary:
1. Batch includes 1 minor planets:
   ['66146']
2. Batch includes 1493 observations, including 50 observations from space telescopes
3. The observations range from 2015-01-10 15:21:46.368010 to 2024-11-14 12:42:12.384011
   In seconds TDB since J2000: 474175373.5521957 to 784860201.5667379
   In Julian Days: 2457033.14012 to 2460629.02931
4. The batch contains observations from 39 observatories, including 1 space telescopes

None


/home/emmabob/miniconda3/envs/tudat-space/lib/python3.10/site-packages/tudatpy/data/mpc/mpc.py:866: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['66146' '66146' '66146' ... '66146' '66146' '66146']' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  obs.loc[:, "number"] = obs.number.astype(str)


In [27]:
# Retrieve the first and final observation epochs and add the buffer
# Manual buffer instead of a random one
epoch_start_nobuffer = DateTime.from_epoch(batch.epoch_start)
epoch_end_nobuffer =  DateTime.from_epoch(batch.epoch_end)

print(f"Epoch Start (no buffer): {epoch_start_nobuffer.to_epoch()}")
print(f"Epoch End (no buffer): {epoch_end_nobuffer.to_epoch()}")

# This samples the cartesian state at 500 points over the observation time:
# I am a bit unsure about this line for now

times_get_eph = np.linspace(epoch_start_nobuffer.to_epoch(), epoch_end_nobuffer.to_epoch(), 500)

epoch_start_buffer = epoch_start_nobuffer.to_epoch() - time_buffer
epoch_end_buffer = epoch_end_nobuffer.to_epoch() + time_buffer

print(f"Epoch Start (buffer): {epoch_start_buffer}")
print(f"Epoch End (buffer): {epoch_end_buffer}")

# # Sure, an initial guess is made here
# I will create it later (here it creates an error when done in this way)
# initial_guess = spice.get_body_cartesian_state_at_epoch(
#     target_spkid,
#     global_frame_origin,
#     global_frame_orientation,
#     "NONE",
#     epoch_start_buffer,
# )

print()
print("Summary of space telescopes in batch:")
print(batch.observatories_table(only_space_telescopes=True))

Epoch Start (no buffer): 474175373.5521957
Epoch End (no buffer): 784860201.5667379
Epoch Start (buffer): 468818573.5521957
Epoch End (buffer): 790217001.5667379

Summary of space telescopes in batch:
     Code  Name  count
1239  C51  WISE   50.0


In [17]:
obs_by_WISE = (
    batch.table.query("observatory == 'C51'")
    .loc[:, ["number", "epochUTC", "RA", "DEC"]]
    .iloc[[0, -1]]
)

print("\nInitial and Final Observations by WISE:")
print(obs_by_WISE)


Initial and Final Observations by WISE:
     number                   epochUTC        RA       DEC
924   66146 2017-08-18 12:40:50.303990  0.967608  0.130601
1153  66146 2019-10-09 17:21:31.392003  5.475997 -0.628676


In [ ]:
# Then in the Tudat example they "Retrieving satellite and astroid ephemerides from JPL Horizons". 
# But since my method instead laods the asteroids into SPICE I will be using my own method
# To extract the positions of the asteroids

# However for the large bodies they still have to be extracted through SPICE directly

In [ ]:
# Set up the environment

# The lagrer bodies exist inside of SPICE and are well-defined
larger_bodies_to_create = [
    "Sun",
    "Earth",
    "Mercury",
    "Venus",
    "Mars",
    "Jupiter",
    "Saturn",
    "Uranus",
    "Neptune"
]

# The 21 smaller bodies do not exist inside of SPICE and therefore their mu has to be manually added
# Create a dictionary to store the names, ID numbers and mu (GM in km3/s2) of the asteroids 

# Will check later if I need to add e.g. radius, radiation or time of orbit around the Sun etc.

smaller_bodies = {
    "Ceres": [1, 62.10], 
    "Pallas": [2, 13.73],
    "Juno": [3, 1.61],
    "Vesta": [4, 17.38],
    "Hebe": [6, 0.89],
    "Iris": [7, 0.73],
    "Flora": [8, 0.27],
    "Metis": [9, 0.44],
    "Hygiea": [10, 5.97],
    "Irene": [14, 0.25],
    "Eunomia": [15, 1.88],
    "Psyche": [16, 1.65],
    "Fortuna": [19, 0.42],
    "Thalia": [23, 0.15],
    "Amphitrite": [29, 0.98],
    "Daphne": [41, 0.56],
    "Europa": [52, 1.48],
    "Bamberga": [324, 0.71],
    "Davida": [511, 1.14],
    "Herculina": [532, 0.66],
    "Interamnia": [704, 2.65]
}



In [ ]:
# Many of the asteroids do not exist in the SPICE kernel
# Therefore I'm now adding them by downloading them in SPK files from the JPL Horizon website
# Then I will extract them with load_kernel

# ---------------------------------------------------------
# The path to the folder where the .bsp files are located
kernel_directory = "/home/emmabob/Bachelor_Project/asteroid_kernels/" 

# Loop through the dictionary and load the corresponding .bsp file
for i, (name, data) in enumerate(smaller_bodies.items(), start=1):
    
    # Calculate the NAIF ID from the data list
    # E.g. Juno has 20000003 (ID: 3)
    ast_id = data[0]
    naif_id = 20000000 + ast_id
    
    # Then extract the ephemeris for each asteroid
    kernel_path = os.path.join(kernel_directory, f"{naif_id}.bsp")
    if os.path.exists(kernel_path):
        spice.load_kernel(kernel_path)
        print(f"{i}.Successfully loaded the kernel for {name} using file: {naif_id}.bsp")
    else:
        print(f"WARNING: Could not find {name}. Its kernel was not found at {kernel_path}. Make sure the file exists.")



In [ ]:
# Extract only the asteroids' names 
# This is so that the list of all bodies can be created
smaller_bodies_list = list(smaller_bodies.keys())


# Manually add 1998 TU3
asteroid_name = "1998-TU3"


# Combine all bodies into one large list
bodies_to_create = larger_bodies_to_create + smaller_bodies_list + [asteroid_name]
bodies_to_propagate = asteroid_name     # I only propagate TU3, I pull the other values from pre-existing ephemeris


# ----------------------------
# Create bodies in simulation.

# Pull on the data already known for the larger bodies
body_settings = environment_setup.get_default_body_settings(
    bodies = larger_bodies_to_create, 
    base_frame_origin = 'SSB',                   # Correct as initial conditions from JPL are in SSB
    base_frame_orientation = 'J2000')       # Intiial conditions from JPL are in J2000

# To avoid the issue of calling for coordinates that don't exist in the ephermeris
# Add a buffer time

buffer_time = 5 * constants.JULIAN_DAY

# A for loop to extract and add settings for the 21 massive asteroids from JPL Horizon

for i, (name, data) in enumerate(smaller_bodies.items(), start=1):

    # Extract asteroid properties
    ast_id = data[0]
    ast_mu = data[1] * 1e9  # Convert km^3/s^2 to m^3/s^2 (SI-units)

    # Calculate the SPICE NAIF ID (2000000 + minor planet number) again
    naif_id = 20000000 + ast_id

    try:
        
        # To fix Tudat finding the data:

        naif_id_str = str(2000000 + ast_id)

        # Instead try extracting the ephemeris from SPICE (w. direct_spice) 
        DIRECT_ephemeris_settings_ast = environment_setup.ephemeris.direct_spice(
            frame_origin = 'Sun',                    # The SPK files were w.r.t the Sun
            frame_orientation = 'J2000',        # tells Tudat the asteroids' coordinates are wrt ECLIPJ2000 
            body_name_to_use = naif_id_str)


        # Create empty slots, then insert the asteroids
        body_settings.add_empty_settings(name)
        # Assign ephemeris
        body_settings.get(name).ephemeris_settings = DIRECT_ephemeris_settings_ast

        # Assign gravity field (w. the central function that manually adds the mu)
        body_settings.get(name).gravity_field_settings = (
            environment_setup.gravity_field.central(ast_mu)
        )

        print(f" {i}. Successfully configured {name} (ID: {ast_id} / NAIF: {naif_id})")

    except Exception as e:
        print(f"Could not fetch data for {name}: {e}")


# Manually add empty settings for TU3
body_settings.add_empty_settings(asteroid_name)

# Know that I create the environment below, when creating body_system


In [ ]:
# Retrieve the asteroid's body name from BatchMPC 
# I am unsure whether this command literally extracts all of the batch..?
# Because this is the first time MPC_objects is mentioned...?
bodies_to_propagate = batch.MPC_objects

# Set its centre to enable its propapgation
# Uhuh, not the Sun...?
central_bodies = ["SSB"] * len(batch.MPC_objects)

In [ ]:
# Create the accelerations
# ^insert mini boss music here^

# Recall:
# Level 1: Sun, 8 planets and GR
# Level 2: Sun, 8 planets, 21 asteroids and GR
# Level 3: Sun, 8 planets, 21 asteroids, GR, J2 and Yarkovsky

In [ ]:
# LVL1

acceleration_LVL1 = {
    "Sun": [
        # The Sun's gravity 
        propagation_setup.acceleration.point_mass_gravity(), 
        # Activating General Relativity (exerted by the Sun / the largest body)
        # Sufficient to use the Schwarzschild Correction
        propagation_setup.acceleration.relativistic_correction(use_schwarzschild=True), # General Relativity
    ],
    "Mercury": [propagation_setup.acceleration.point_mass_gravity()],
    "Venus": [propagation_setup.acceleration.point_mass_gravity()],
    "Earth": [propagation_setup.acceleration.point_mass_gravity()],
    "Mars": [propagation_setup.acceleration.point_mass_gravity()],
    "Jupiter": [propagation_setup.acceleration.point_mass_gravity()],
    "Saturn": [propagation_setup.acceleration.point_mass_gravity()],
    "Uranus": [propagation_setup.acceleration.point_mass_gravity()],
    "Neptune": [propagation_setup.acceleration.point_mass_gravity()],
}

acceleration_dict1 = {"1998-TU3": acceleration_LVL1}

In [ ]:
# LVL2

acceleration_planets_LVL2 = {
    "Sun": [
        # The Sun's gravity 
        propagation_setup.acceleration.point_mass_gravity(), 
        # Activating General Relativity (exerted by the Sun / the largest body)
        # Sufficient to use the Schwarzschild Correction
        propagation_setup.acceleration.relativistic_correction(use_schwarzschild=True), # General Relativity
    ],
    "Mercury": [propagation_setup.acceleration.point_mass_gravity()],
    "Venus": [propagation_setup.acceleration.point_mass_gravity()],
    "Earth": [propagation_setup.acceleration.point_mass_gravity()],
    "Mars": [propagation_setup.acceleration.point_mass_gravity()],
    "Jupiter": [propagation_setup.acceleration.point_mass_gravity()],
    "Saturn": [propagation_setup.acceleration.point_mass_gravity()],
    "Uranus": [propagation_setup.acceleration.point_mass_gravity()],
    "Neptune": [propagation_setup.acceleration.point_mass_gravity()],
}

# The asteroids
accelerations_asteroids = {
    
    asteroid_name: [
        propagation_setup.acceleration.point_mass_gravity()
    ]
    for asteroid_name in smaller_bodies_list
}

# Merge dictionaries
accelerations_merged_LVL2 = (
    acceleration_planets_LVL2 |
    accelerations_asteroids
)

# Finally, define the acceleration for TU3
# Because only TU3 is propagated
acceleration_dict2 = {
    "1998-TU3": accelerations_merged_LVL2
}



In [ ]:
# LVL3
# Please see explanation in TU3_orbit_precession_2.ipynb file

In [ ]:
# Define the solar quadrupole moment
# See Book by Montenbruck and Gill, pg 57-58 for the conversion/ equations
# To understand function see: https://py.api.tudat.space/en/latest/dynamics/environment_setup/gravity_field.html#tudatpy.dynamics.environment_setup.gravity_field.spherical_harmonic

# For the Sun's gravitational potential, Sn0 is zero by definition

J_2 = 2.2 * 10**(-7)
norm_C_20 = - J_2 / np.sqrt(5)

norm_cosine_coeffs = np.array([
    [1.0, 0.0, 0.0],
    [0.0, 0.0, 0.0],
    [norm_C_20, 0.0, 0.0]
])

norm_sine_coeffs = np.array([
    [0.0, 0.0, 0.0],
    [0.0, 0.0, 0.0],
    [0.0, 0.0, 0.0]
])

# Technically I've defined this below as well
mu_Sun = 1.327124400420322e+20
radius_Sun = 695.99 * 10**6         # ± 0.07 * 10**6 m

body_settings.get("Sun").gravity_field_settings = (
    environment_setup.gravity_field.spherical_harmonic(
        gravitational_parameter = mu_Sun,
        reference_radius = radius_Sun,
        normalized_cosine_coefficients = norm_cosine_coeffs,
        normalized_sine_coefficients = norm_sine_coeffs,
        # Ensure Tudat knows that the values are defined in the Sun's fixed frame
        # The International Astronomical Union (IAU) defines standard body-fixed frames for well-known bodies
        associated_reference_frame = "IAU_Sun"      # Ensures that the buldge rotates with the body
    )
)

In [ ]:
# For TU3 the value of A2 is computed 

avg_a_TU3_greenberg = -5.60 * 10**(-4)                       # AU / Myr 
avg_a_TU3_calc = -5.60 * 10**(-4) / (10**6 * 365)            # AU / days
uncertain_avg_a_TU3_greenberg = 3.9 * 10**(-4)               # AU / Mry
uncertain_avg_a_TU3_calc = 3.9 * 10**(-4) / (10**6 * 365)    # AU / days

a_TU3_calc = 0.7875484323220899 # AU
e_TU3_calc = 0.4836694929440215 # unitless
n_TU3_calc = 1.410224212386279  # degrees/ days (remember that angles are unitless in unit conversion)

A_2_TU3_AUdays2 = avg_a_TU3_calc * (n_TU3_calc * (a_TU3_calc**2) * (1 - (e_TU3_calc**2))) / 2
A_2_TU3_uncertainty_AUdays2 = uncertain_avg_a_TU3_calc * (n_TU3_calc * (a_TU3_calc**2) * (1 - (e_TU3_calc**2))) / 2

print(f"The Yarkovksy parameter for TU3: {A_2_TU3_AUdays2} AU / days^2")
print(A_2_TU3_uncertainty_AUdays2)

# Units from AU/days2 to m/s2
A_2_TU3_ms2 = A_2_TU3_AUdays2 * 149597870691 / ((24 * 3600)**2)
print(A_2_TU3_ms2)

In [ ]:
# The planets
acceleration_planets_LVL3 = {
    "Sun": [
        # The Sun's gravity 
        # propagation_setup.acceleration.point_mass_gravity(), 
        # A spherical harmonic (not just a point mass, but also the solar quadrupole moment)
        propagation_setup.acceleration.spherical_harmonic_gravity(2, 0) ,
        # Activating General Relativity (exerted by the Sun / the largest body)
        # Sufficient to use the Schwarzschild Correction
        propagation_setup.acceleration.relativistic_correction(use_schwarzschild=True), # General Relativity
        # Yarkovsky correction
        propagation_setup.acceleration.yarkovsky(A_2_TU3_AUdays2), 
    ],
    "Mercury": [propagation_setup.acceleration.point_mass_gravity()],
    "Venus": [propagation_setup.acceleration.point_mass_gravity()],
    "Earth": [propagation_setup.acceleration.point_mass_gravity()],
    "Mars": [propagation_setup.acceleration.point_mass_gravity()],
    "Jupiter": [propagation_setup.acceleration.point_mass_gravity()],
    "Saturn": [propagation_setup.acceleration.point_mass_gravity()],
    "Uranus": [propagation_setup.acceleration.point_mass_gravity()],
    "Neptune": [propagation_setup.acceleration.point_mass_gravity()],
}

# Merge dictionaries
accelerations_merged_LVL3 = (
    acceleration_planets_LVL3 |
    accelerations_asteroids
)

# Finally, define the acceleration for TU3
# Because only TU3 is propagated
acceleration_dict3 = {
    "1998-TU3": accelerations_merged_LVL3
}

In [ ]:
# Dictionary with the three acceleration setting options
acceleration_sets = {1: acceleration_dict1, 2: acceleration_dict2, 3: acceleration_dict3}


In [ ]:
# Thus, the environment becomes:
body_system = environment_setup.create_system_of_bodies(body_settings)

In [ ]:
# Step ...
# Initial condition / guess for TU3

sun_gravitational_parameter = body_system.get("Sun").gravitational_parameter

# Grabbing the initial state at start_epoch from JPL with cartesian_to_keplarian() (in astro)

# Cartesian state vector
# I manually change this in JPL

# For J2000
# cartesian_elements = np.array([
#     4.543863572576185E+07 * 1e3,   # X  [m]
#     -9.673019712424231E+07 * 1e3,  # Y  [m]
#     -2.313075952057116E+06 * 1e3,  # Z  [m]
#     3.661410298575712E+01 * 1e3,   # VX [m/s]
#     -1.757686929389374E+00 * 1e3,  # VY [m/s]
#     -3.350242409196601E+00 * 1e3   # VZ [m/s]
# ], dtype=np.float64)

# For 2015-01-01 at 12:00:00 according to JPL
cartesian_elements = np.array([
    1.507125751810296E+08 * 1e3,   # X  [m]
    7.065187901784344E+07 * 1e3,  # Y  [m]
    -1.535599772350254E+07 * 1e3,  # Z  [m]
    -1.403410074676226E+01 * 1e3,   # VX [m/s]
    1.631479524621320E+01 * 1e3,  # VY [m/s]
    9.734115926434903E-01 * 1e3   # VZ [m/s]
], dtype=np.float64)

test_initial_TU3_array = element_conversion.cartesian_to_keplerian(
    cartesian_elements = cartesian_elements,
    gravitational_parameter = sun_gravitational_parameter
    )

print("Keplerian Elements:")
print(test_initial_TU3_array)
# print(test_initial_TU3_array[0])
# print(test_initial_TU3_array[2])

# ---------------------------------
# The initial state is thus:

initial_state_TU3 = element_conversion.keplerian_to_cartesian_elementwise(
    gravitational_parameter = sun_gravitational_parameter,
    semi_major_axis = test_initial_TU3_array[0],                 #meters
    eccentricity = test_initial_TU3_array[1],                    #unitless
    inclination = test_initial_TU3_array[2],                     # cartesian_to_keplerian returns angles in radians
    argument_of_periapsis = test_initial_TU3_array[3],
    longitude_of_ascending_node = test_initial_TU3_array[4],
    true_anomaly = test_initial_TU3_array[5],                
)

In [ ]:
# Add random offset for initial guess
rng = np.random.default_rng(seed=1)

initial_position_offset = 1e6 * 1000
initial_velocity_offset = 100

initial_guess = initial_state_TU3.copy()
initial_guess[0:3] += (2 * rng.random(3) - 1) * initial_position_offset
initial_guess[3:6] += (2 * rng.random(3) - 1) * initial_velocity_offset

print("Error between the real initial state and our initial guess:")
print(initial_guess - initial_state_TU3)

In [ ]:
# Create numerical integrator settings
integrator_settings = propagation_setup.integrator.runge_kutta_variable_step_size(
    timestep_global,
    propagation_setup.integrator.CoefficientSets.rkf_78,
    timestep_global,
    timestep_global,
    1.0,
    1.0,
)
# Terminate at the time of oldest observation
termination_condition = propagation_setup.propagator.time_termination(epoch_end_buffer)

In [ ]:
# To propagate:

# Directly select acceleration settings
acceleration_settings = acceleration_sets[acceleration_level]

# Select acceleration model set
acceleration_settings = acceleration_sets[acceleration_level]

# Create acceleration models
acceleration_models = propagation_setup.create_acceleration_models(
    bodies,
    acceleration_settings,
    bodies_to_propagate,
    central_bodies,
)

In [ ]:
def perform_estimation(
        bodies,
        acceleration_level:int,
        use_satellite_data: bool,
        apply_star_catalog_debias: bool,
        apply_weighting_scheme: bool,
):
    # The satellites are present in the integration of all setups,
    # the included satellitess parameter in to_tudat() dictates whether a satellite's observations are used.
    if use_satellite_data:
        included_satellites = {
            mpc: name for mpc, name in zip(satellites_MPC_codes, satellites_names)
        }
    else:
        included_satellites = None

    # As in the first example, the observation collection is created with BatchMPC.to_tudat()
    # This time, the star catalog biases and weights are enabled,
    # the included_satellites parameter ensures satellite observations are included.
    # internally, to_tudat() links a space telescope's observatory code to the spacecraft's dynamics.
    batch_temp = batch.copy()
    observation_collection = batch_temp.to_tudat(
        bodies=bodies,
        included_satellites=included_satellites,
        apply_star_catalog_debias=apply_star_catalog_debias,
        apply_weights_VFCC17=apply_weighting_scheme,
    )

    # Set up the accelerations settings for each body, in this case only Eros
    acceleration_settings = {}
    for body in bodies_to_propagate:
        acceleration_settings[str(body)] = acceleration_sets[acceleration_level]

    # Create the acceleration models.
    acceleration_models = propagation_setup.create_acceleration_models(
        bodies, acceleration_settings, bodies_to_propagate, central_bodies
    )

    # Set create angular_position settings for each link in the list.
    observation_settings_list = list()
    link_list = list(
        observation_collection.get_link_definitions_for_observables(
            observable_type=observable_models_setup.model_settings.angular_position_type
        )
    )
    for link in link_list:
        observation_settings_list.append(
            observable_models_setup.model_settings.angular_position(link, bias_settings=None)
        )

    # Create propagation settings
    propagator_settings = propagation_setup.propagator.translational(
        central_bodies=central_bodies,
        acceleration_models=acceleration_models,
        bodies_to_integrate=bodies_to_propagate,
        initial_states=initial_guess,
        initial_time=epoch_start_buffer,
        integrator_settings=integrator_settings,
        termination_settings=termination_condition,
    )

    # Setup parameters settings to propagate the state transition matrix
    parameter_settings = parameters_setup.initial_states(
        propagator_settings, bodies
    )

    # Create the parameters that will be estimated
    parameters_to_estimate = parameters_setup.create_parameter_set(
        parameter_settings, bodies, propagator_settings
    )

    # Set up the estimator
    estimator = estimation_analysis.Estimator(
        bodies=bodies,
        estimated_parameters=parameters_to_estimate,
        observation_settings=observation_settings_list,
        propagator_settings=propagator_settings,
        integrate_on_creation=True,
    )

    # provide the observation collection as input, and limit number of iterations for estimation.
    pod_input = estimation_analysis.EstimationInput(
        observations_and_times=observation_collection,
        convergence_checker=estimation_analysis.estimation_convergence_checker(
            maximum_iterations=number_of_pod_iterations,
        ),
    )

    # to_tudat() applies weights to a set of observations between an observatory and the target.
    # the method below tells tudat to use the weights applied to these sets.
    # This step is required when setting weights through the BatchMPC class.
    if apply_weighting_scheme:
        pod_input.set_weights_from_observation_collection()

    # Set methodological options
    pod_input.define_estimation_settings(reintegrate_variational_equations=True)

    # Perform the estimation
    pod_output = estimator.perform_estimation(pod_input)

    # we store the following outputs for plotting and analysis.
    return pod_output, batch_temp, observation_collection, estimator